# Exploración inicial de las fuentes

**Proyecto:** Evolución del precio de la vivienda en Asturias.

Este notebook recoge las comprobaciones realizadas sobre los cuatro archivos originales: IPV, IPC, salarios y valor tasado. Su finalidad es conocer la estructura, cobertura y calidad de los datos antes de limpiarlos.

**Criterio:** no modificar los archivos de `data/raw/`. Las interpretaciones y decisiones metodológicas se documentan en `docs/informe_exploracion.md`.

## 1. Importación y localización de archivos

In [1]:
from pathlib import Path
import re
import warnings
import pandas as pd

warnings.filterwarnings(
    "ignore",
    message="Workbook contains no default style",
    category=UserWarning,
)

# Funciona al ejecutarlo desde notebooks/ o desde la raíz del repositorio.
raiz = Path.cwd()
if not (raiz / "data" / "raw").is_dir():
    raiz = raiz.parent

raw = raiz / "data" / "raw"
assert raw.is_dir(), "No se encuentra la carpeta data/raw del proyecto."

nombres_archivos = {
    "IPV": "base25_raw.xlsx",
    "IPC": "ipcgrupos_raw.xlsx",
    "Salarios": "salarios_raw.xlsx",
    "Valor tasado": "tasa_raw.csv",
}

rutas = {}
for fuente, nombre in nombres_archivos.items():
    encontrados = sorted(raw.rglob(nombre))
    if not encontrados:
        print(f"NO ENCONTRADO: {fuente} ({nombre})")
    else:
        rutas[fuente] = encontrados[0]
        print(f"{fuente}: {encontrados[0].relative_to(raiz)}")

IPV: data\raw\ipv\base25_raw.xlsx
IPC: data\raw\ipc\ipcgrupos_raw.xlsx
Salarios: data\raw\nucleo\salarios_raw.xlsx
Valor tasado: data\raw\nucleo\tasa_raw.csv


## 2. Carga y estructura de las tablas

Se leen los Excel con `header=None` para conservar las cabeceras originales como filas. El CSV de tasación utiliza `;` como separador.

In [2]:
datos = {}
for fuente, ruta in rutas.items():
    if ruta.suffix.lower() == ".xlsx":
        datos[fuente] = pd.read_excel(ruta, header=None)
    else:
        datos[fuente] = pd.read_csv(ruta, sep=";", encoding="utf-8-sig", low_memory=False)

resumen_estructura = pd.DataFrame([
    {"Fuente": fuente, "Filas": len(df), "Columnas": len(df.columns),
     "Nulos en hoja completa": int(df.isna().sum().sum())}
    for fuente, df in datos.items()
])
display(resumen_estructura)

,Fuente,Filas,Columnas,Nulos en hoja completa
0,IPV,94,313,10297
1,IPC,314,1189,39205
2,Salarios,50,19,581
3,Valor tasado,13711,8,4805


In [3]:
for fuente, df in datos.items():
    print(f"\n{fuente}: primeras filas y columnas")
    display(df.iloc[:12, :min(6, df.shape[1])])
    print("Tipos de datos por columna (primeras 10):")
    display(df.dtypes.head(10))


IPV: primeras filas y columnas


,0,1,2,3,4,5
0,Índice de Precios de Vivienda. Base 2025,NaN,NaN,NaN,NaN,NaN
1,Índice de Precios de Vivienda (IPV). Base 2025,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN
3,"Índices por CCAA: general, vivienda nueva y de...",NaN,NaN,NaN,NaN,NaN
4,"Unidades: Índice, Tasas",NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,NaN
6,,Índice,NaN,NaN,NaN,NaN
7,,2026T2,2026T1,2025T4,2025T3,2025T2
8,Nacional,NaN,NaN,NaN,NaN,NaN
9,General,111.095,107.458,103.805,101.937,99.043


Tipos de datos por columna (primeras 10):


0       str
1    object
2    object
3    object
4    object
5    object
6    object
7    object
8    object
9    object
dtype: object


IPC: primeras filas y columnas


,0,1,2,3,4,5
0,Indice de Precios de Consumo. Base 2025,NaN,NaN,NaN,NaN,NaN
1,Resultados por comunidades autónomas,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN
3,Índices por comunidades autónomas: general y d...,NaN,NaN,NaN,NaN,NaN
4,"Unidades: Índice, Tasas",NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,NaN
6,,Índice,NaN,NaN,NaN,NaN
7,,2026M09,2026M08,2026M07,2026M06,2026M05
8,Nacional,NaN,NaN,NaN,NaN,NaN
9,Índice general,104.978,104.638,103.899,103.598,102.951


... (6 outputs left out)


Valor tasado: primeras filas y columnas


,Año,Trimestre,Valor,Régimen,CPRO,Provincia
0,1995,1,NaN,Libre,NaN,Ceuta y Melilla
1,1995,2,NaN,Libre,NaN,Ceuta y Melilla
2,1995,3,NaN,Libre,NaN,Ceuta y Melilla
3,1995,4,NaN,Libre,NaN,Ceuta y Melilla
4,1996,1,NaN,Libre,NaN,Ceuta y Melilla
5,1996,2,NaN,Libre,NaN,Ceuta y Melilla
6,1996,3,NaN,Libre,NaN,Ceuta y Melilla
7,1996,4,NaN,Libre,NaN,Ceuta y Melilla
8,1997,1,NaN,Libre,NaN,Ceuta y Melilla
9,1997,2,NaN,Libre,NaN,Ceuta y Melilla


Tipos de datos por columna (primeras 10):


Año                     int64
Trimestre               int64
Valor                 float64
Régimen                   str
CPRO                  float64
Provincia                 str
Comunidad_Autónoma        str
CODAUTO               float64
dtype: object

## 3. Identificación de medidas, categorías y territorios

En el IPV y el IPC, la fila 7 de Excel contiene los nombres de los bloques de medidas y la fila 8, sus periodos. En salarios se inspeccionan las primeras filas y sus cabeceras territoriales. El CSV del valor tasado tiene nombres de columnas propios.

In [4]:
for fuente in ("IPV", "IPC"):
    if fuente not in datos:
        continue
    df = datos[fuente]
    fila_medidas = df.iloc[6]
    bloques = [(i, str(v)) for i, v in enumerate(fila_medidas) if pd.notna(v)]
    print(f"\n{fuente} — bloques de medidas (posición de columna empezando en 0):")
    display(pd.DataFrame(bloques, columns=["Columna", "Medida"]))
    print("Etiquetas territoriales y de categorías (primeras 65 filas):")
    display(df.iloc[8:65, [0]])


IPV — bloques de medidas (posición de columna empezando en 0):


,Columna,Medida
0,0,
1,1,Índice
2,79,Variación trimestral
3,157,Variación anual
4,235,Variación en lo que va de año


Etiquetas territoriales y de categorías (primeras 65 filas):


,0
8,Nacional
9,General
10,Vivienda nueva
11,Vivienda segunda mano
12,01 Andalucía
13,General
14,Vivienda nueva
15,Vivienda segunda mano
16,02 Aragón
17,General



IPC — bloques de medidas (posición de columna empezando en 0):


,Columna,Medida
0,0,
1,1,Índice
2,298,Variación mensual
3,595,Variación anual
4,892,Variación en lo que va de año


Etiquetas territoriales y de categorías (primeras 65 filas):


,0
8,Nacional
9,Índice general
10,01 Alimentos y bebidas no alcohólicas
11,02 Bebidas alcohólicas y tabaco
12,03 Vestido y calzado
13,"04 Vivienda, agua, electricidad, gas y otr..."
14,"05 Muebles, artículos del hogar y artículo..."
15,06 Sanidad
16,07 Transporte
17,08 Información y comunicaciones


In [5]:
if "Salarios" in datos:
    df = datos["Salarios"]
    print("Salarios — primeras 14 filas y 8 columnas")
    display(df.iloc[:14, :min(8, df.shape[1])])
    print("Cabeceras territoriales (fila Excel 7):")
    display(df.iloc[6].dropna())

Salarios — primeras 14 filas y 8 columnas


,0,1,2,3,4,5,6,7
0,Resultados nacionales y por comunidades aútonomas,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Ganancia media anual por trabajador,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Sexo y sectores de actividad,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Unidades: €,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,,Total Nacional,Andalucía,Aragón,"Asturias, Principado de","Balears, Illes",Canarias,Cantabria
7,Todos los sectores,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,2024,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,Ambos sexos,29540.26,26089.7,28061.94,28562.31,29075.06,25120.38,27097.17


Cabeceras territoriales (fila Excel 7):


0                                
1                  Total Nacional
2                       Andalucía
3                          Aragón
4         Asturias, Principado de
5                  Balears, Illes
6                        Canarias
7                       Cantabria
8                 Castilla y León
9            Castilla - La Mancha
10                       Cataluña
11           Comunitat Valenciana
12                    Extremadura
13                        Galicia
14           Madrid, Comunidad de
15              Murcia, Región de
16    Navarra, Comunidad Foral de
17                     País Vasco
18                      Rioja, La
Name: 6, dtype: object

In [6]:
if "Valor tasado" in datos:
    df = datos["Valor tasado"]
    print("Columnas:")
    print(df.columns.tolist())
    for columna in ["Régimen", "Provincia", "Comunidad_Autónoma"]:
        if columna in df:
            print(f"\n{columna} — valores de ejemplo:")
            display(df[columna].value_counts(dropna=False).head(15))

Columnas:
['Año', 'Trimestre', 'Valor', 'Régimen', 'CPRO', 'Provincia', 'Comunidad_Autónoma', 'CODAUTO']

Régimen — valores de ejemplo:


Régimen
Libre        8174
Protegida    5537
Name: count, dtype: int64


Provincia — valores de ejemplo:


Provincia
Total CCAA         3323
Ceuta y Melilla     212
Valladolid          212
León                212
Burgos              212
Salamanca           212
Zamora              212
Palencia            212
Ávila               212
Segovia             212
Soria               212
Jaén                212
Córdoba             212
Málaga              212
Cádiz               212
Name: count, dtype: int64


Comunidad_Autónoma — valores de ejemplo:


Comunidad_Autónoma
Castilla y León                2120
Andalucía                      1908
Castilla-La Mancha             1272
Galicia                        1060
Cataluña                       1060
Aragón                          848
Comunitat Valenciana            848
País Vasco                      848
Extremadura                     636
Canarias                        636
Navarra, Comunidad Foral de     354
Ceuta y Melilla                 212
Balears, Illes                  212
Cantabria                       212
Ceuta                           212
Name: count, dtype: int64

## 4. Cobertura temporal y geográfica

Se comprueban los periodos observados y la presencia de Asturias y del agregado nacional. Los periodos de los Excel se extraen de las cabeceras, no de los valores.

In [7]:
patron_periodo = re.compile(r"^(?:19|20)\d{2}[TM]\d{1,2}$")

for fuente in ("IPV", "IPC"):
    if fuente not in datos:
        continue
    df = datos[fuente]
    periodos = sorted({
        str(valor).strip()
        for valor in df.iloc[7].dropna()
        if patron_periodo.fullmatch(str(valor).strip())
    })
    etiquetas = df.iloc[:, 0].dropna().astype(str)
    print(f"\n{fuente}: {len(periodos)} periodos distintos")
    print("Desde:", periodos[0] if periodos else "No detectado")
    print("Hasta:", periodos[-1] if periodos else "No detectado")
    print("Referencias a Asturias:", etiquetas.str.contains("Asturias", case=False).sum())
    print("Referencias a Nacional/España:",
          etiquetas.str.contains("Nacional|España", case=False, regex=True).sum())


IPV: 78 periodos distintos
Desde: 2007T1
Hasta: 2026T2
Referencias a Asturias: 1
Referencias a Nacional/España: 2

IPC: 297 periodos distintos
Desde: 2002M01
Hasta: 2026M09
Referencias a Asturias: 1
Referencias a Nacional/España: 2


In [8]:
if "Salarios" in datos:
    df = datos["Salarios"]
    print("Salarios — filas y columnas donde se menciona Asturias o Nacional:")
    mask = df.astype(str).apply(
        lambda col: col.str.contains("Asturias|Nacional|España", case=False, regex=True, na=False)
    )
    posiciones = list(zip(*mask.to_numpy().nonzero()))
    print(posiciones[:30])
    print("Últimas filas para examinar el periodo:")
    display(df.iloc[-12:, :min(6, df.shape[1])])

Salarios — filas y columnas donde se menciona Asturias o Nacional:
[(np.int64(0), np.int64(0)), (np.int64(6), np.int64(1)), (np.int64(6), np.int64(4)), (np.int64(49), np.int64(0))]
Últimas filas para examinar el periodo:


,0,1,2,3,4,5
38,2009,NaN,NaN,NaN,NaN,NaN
39,Ambos sexos,22511.47,20906.33,22133.77,22597.97,21070.14
40,2008,NaN,NaN,NaN,NaN,NaN
41,Ambos sexos,21883.42,20138.56,21594.47,21805.67,20388.09
42,NaN,NaN,NaN,NaN,NaN,NaN
43,NaN,NaN,NaN,NaN,NaN,NaN
44,Notas:,NaN,NaN,NaN,NaN,NaN
45,El dato no esta disponible cuando el número de...,NaN,NaN,NaN,NaN,NaN
46,Cuando la casilla está marcada con un signo '-...,NaN,NaN,NaN,NaN,NaN
47,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
if "Valor tasado" in datos:
    df = datos["Valor tasado"]
    for col in ["Año", "Trimestre"]:
        if col in df:
            print(f"{col}: mínimo={df[col].min()}, máximo={df[col].max()}")
    if "Comunidad_Autónoma" in df:
        print("Registros asociados a Asturias:",
              df["Comunidad_Autónoma"].astype(str).str.contains("Asturias", case=False).sum())
    if "Provincia" in df:
        print("Agregados territoriales (ejemplos):")
        display(df["Provincia"].value_counts(dropna=False).head(12))

Año: mínimo=1995, máximo=2026
Trimestre: mínimo=1, máximo=4
Registros asociados a Asturias: 212
Agregados territoriales (ejemplos):


Provincia
Total CCAA         3323
Ceuta y Melilla     212
Valladolid          212
León                212
Burgos              212
Salamanca           212
Zamora              212
Palencia            212
Ávila               212
Segovia             212
Soria               212
Jaén                212
Name: count, dtype: int64

## 5. Comprobaciones de calidad

Los recuentos de nulos de los Excel incluyen celdas estructurales vacías. Por tanto, no deben interpretarse directamente como valores faltantes en una serie. Tampoco todos los registros aparentemente repetidos son errores.

In [10]:
calidad = pd.DataFrame([
    {"Fuente": fuente,
     "Celdas vacías (total hoja)": int(df.isna().sum().sum()),
     "Filas duplicadas exactas": int(df.duplicated().sum())}
    for fuente, df in datos.items()
])
display(calidad)

,Fuente,Celdas vacías (total hoja),Filas duplicadas exactas
0,IPV,10297,6
1,IPC,39205,4
2,Salarios,581,4
3,Valor tasado,4805,142


In [11]:
if "Valor tasado" in datos:
    tasacion = datos["Valor tasado"]
    print("Nulos por columna del valor tasado:")
    display(pd.DataFrame({
        "Nulos": tasacion.isna().sum(),
        "Porcentaje": (tasacion.isna().mean() * 100).round(2),
        "Tipo inferido": tasacion.dtypes.astype(str),
    }))
    claves = [c for c in ["Año", "Trimestre", "Régimen", "CPRO", "Provincia",
                         "Comunidad_Autónoma", "CODAUTO"] if c in tasacion]
    if claves:
        print("Posibles registros repetidos según claves territoriales y temporales:",
              tasacion.duplicated(subset=claves).sum())
        print("Es solo una comprobación; requiere interpretar los niveles territoriales.")

Nulos por columna del valor tasado:


,Nulos,Porcentaje,Tipo inferido
Año,0,0.00,int64
Trimestre,0,0.00,int64
Valor,844,6.16,float64
Régimen,0,0.00,str
CPRO,3535,25.78,float64
Provincia,0,0.00,str
Comunidad_Autónoma,1,0.01,str
CODAUTO,425,3.10,float64


Posibles registros repetidos según claves territoriales y temporales: 142
Es solo una comprobación; requiere interpretar los niveles territoriales.


In [12]:
if "Salarios" in datos:
    salarios = datos["Salarios"].astype(str)
    marcadores = salarios.apply(
        lambda col: col.str.match(r"^\s*-\s*\d", na=False)
    )
    print("Celdas de salarios con signo '-' seguido de cifra:", int(marcadores.sum().sum()))
    print("Revisar la nota de fiabilidad estadística del INE; no convertirlas en salarios negativos.")

Celdas de salarios con signo '-' seguido de cifra: 0
Revisar la nota de fiabilidad estadística del INE; no convertirlas en salarios negativos.


## 6. Síntesis de la exploración

| Fuente | Estructura identificada | Preparación necesaria |
|---|---|---|
| **IPV** | Bloques de medidas por columnas; periodos trimestrales en la fila 8; territorios en bloques de cuatro filas | Identificar medida, trimestre, territorio y tipo de vivienda; pasar a formato largo |
| **IPC** | Bloques de medidas por columnas; periodos mensuales en la fila 8; territorios en bloques de quince filas | Identificar medida, mes, territorio y grupo ECOICOP; pasar a formato largo |
| **Salarios** | Territorios distribuidos por columnas; años y datos salariales en filas | Separar año y salarios; conservar la información territorial y tratar marcas de fiabilidad |
| **Valor tasado** | CSV estructurado con periodo, territorio, régimen y valor | Distinguir agregados territoriales, revisar nulos y duplicados por clave |

**Alcance de este notebook:** exploración y comprobaciones preliminares. No se modifican los ficheros originales, ni se eliminan datos. Las reglas de limpieza y las conclusiones interpretativas se recogen en `docs/informe_exploracion.md`.